In [3]:
import pandas as pd
df = pd.read_csv("/content/Reviews.csv", on_bad_lines='skip', engine='python')

df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [4]:
df.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,13
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,9
Text,0


In [5]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

# Convert Ratings to Sentiment Labels

In [6]:
def sentiment_label(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

df['Sentiment'] = df['Score'].apply(sentiment_label)

df[['Score', 'Sentiment']].head()

,Score,Sentiment
0,5,positive
1,1,negative
2,4,positive
3,2,negative
4,5,positive


In [7]:
df = df[['Text', 'Sentiment']]

df.head()

,Text,Sentiment
0,I have bought several of the Vitality canned d...,positive
1,Product arrived labeled as Jumbo Salted Peanut...,negative
2,This is a confection that has been around a fe...,positive
3,If you are looking for the secret ingredient i...,negative
4,Great taffy at a great price. There was a wid...,positive


# Check Class Distribution

In [8]:
df['Sentiment'].value_counts()

,count
Sentiment,
positive,173567
negative,32342
neutral,17219


# Reduce Dataset Size

In [9]:
df = df.sample(20000, random_state=42)

In [10]:
df = df.reset_index(drop=True)

# Basic Text Cleaning

In [11]:
!pip install nltk

In [12]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [14]:
df['Cleaned_Text'] = df['Text'].apply(clean_text)

df.head()

,Text,Sentiment,Cleaned_Text
0,This is an amazing syrup. It is sweet and it a...,positive,amazing syrup sweet adds exquisite taste enhan...
1,"The coffee has a very nice, chocolaty taste, v...",positive,coffee nice chocolaty taste smooth drawback pa...
2,it's not your typical pretzel. I don't eat muc...,neutral,typical pretzel dont eat much pumpernickel too...
3,"When I bought this popcorn, it was because Ama...",positive,bought popcorn amazon recommended least showed...
4,I did not buy this at Amazon even though I am ...,positive,buy amazon even though avid amazon shopper nee...


# Encode Labels

In [15]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['Sentiment'])

df.head()

,Text,Sentiment,Cleaned_Text,label
0,This is an amazing syrup. It is sweet and it a...,positive,amazing syrup sweet adds exquisite taste enhan...,2
1,"The coffee has a very nice, chocolaty taste, v...",positive,coffee nice chocolaty taste smooth drawback pa...,2
2,it's not your typical pretzel. I don't eat muc...,neutral,typical pretzel dont eat much pumpernickel too...,1
3,"When I bought this popcorn, it was because Ama...",positive,bought popcorn amazon recommended least showed...,2
4,I did not buy this at Amazon even though I am ...,positive,buy amazon even though avid amazon shopper nee...,2


# Train Test Split

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [17]:
train_data = pd.DataFrame({
    'text': X_train,
    'label': y_train
})

test_data = pd.DataFrame({
    'text': X_test,
    'label': y_test
})

train_data.to_csv("train_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)

In [18]:
from google.colab import files

files.download("train_data.csv")
files.download("test_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>